# Mapeamento da produção científica com PLN (SciELO)

Notebook da análise: preparação dos resumos, análise linguística com spaCy, termos e tendências, modelagem de tópicos e cautela científica (*hedging*).

Sem os dados reais, o notebook roda com a amostra sintética `DATA/artigos_sinteticos.csv` (resumos inventados).

## 1. Configuração

In [ ]:
from pathlib import Path

# Arquivo exportado pelo Octoparse (ou a amostra sintética para testes)
ARQUIVO = Path('../DATA/artigos_sinteticos.csv')

ANO_INICIAL, ANO_FINAL = 2015, 2024
N_TOPICOS = 3          # ajuste conforme o tamanho da base (ex.: 8 a 15)
MODELO_SPACY = 'pt_core_news_lg'

PASTA_RESULTADOS = Path('../DATA/resultados')
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from spacy.matcher import Matcher
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

try:
    nlp = spacy.load(MODELO_SPACY)
except OSError:
    # Modelo menor, caso o grande não esteja instalado
    nlp = spacy.load('pt_core_news_sm')
print('Modelo:', nlp.meta['name'])

## 2. Preparação dos dados

In [ ]:
df = pd.read_csv(ARQUIVO, dtype={'ano': 'Int64'})
n_inicial = len(df)

df['resumo'] = (df['resumo'].fillna('')
                .str.replace(r'\s+', ' ', regex=True)
                .str.strip())
df['doi'] = df['doi'].str.lower().str.replace(r'^https?://(dx\.)?doi\.org/', '', regex=True)

df = (df[df['resumo'].str.len() > 0]
      .query('idioma == "pt" and @ANO_INICIAL <= ano <= @ANO_FINAL')
      .drop_duplicates(subset='doi')
      .reset_index(drop=True))

print(f'{n_inicial} artigos coletados → {len(df)} após a limpeza')
df[['ano', 'periodico', 'titulo']].head()

## 3. Análise linguística (spaCy)

Lematização, classes gramaticais, sintagmas nominais e entidades nomeadas.

In [ ]:
CLASSES = {'NOUN', 'PROPN', 'ADJ'}
# Palavras comuns a quase todo resumo, que não indicam tema
STOP_ACADEMICAS = {'estudo', 'objetivo', 'resultado', 'análise', 'método', 'trabalho',
                   'pesquisa', 'dado', 'participante', 'amostra', 'presente'}

def processar(doc):
    lemas = [t.lemma_.lower() for t in doc
             if t.pos_ in CLASSES and not t.is_stop and t.is_alpha and len(t) > 2
             and t.lemma_.lower() not in STOP_ACADEMICAS]
    sintagmas = []
    for sn in doc.noun_chunks:
        tokens = [t for t in sn if not (t.is_stop and t.i == sn.start)]  # tira o artigo inicial
        if len(tokens) >= 2:
            sintagmas.append(' '.join(t.text.lower() for t in tokens))
    entidades = [(e.text, e.label_) for e in doc.ents]
    return lemas, sintagmas, entidades, len(doc)

docs = list(nlp.pipe(df['resumo']))
df[['lemas', 'sintagmas', 'entidades', 'n_tokens']] = pd.DataFrame(
    [processar(d) for d in docs], index=df.index)
df[['titulo', 'lemas', 'sintagmas']].head(3)

## 4. Termos e tendências

In [ ]:
termos = df.explode('lemas').groupby('lemas').size().sort_values(ascending=False)
sintagmas = df.explode('sintagmas').groupby('sintagmas').size().sort_values(ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
termos.head(15).sort_values().plot.barh(ax=ax[0], title='Termos mais frequentes (lemas)')
sintagmas.head(15).sort_values().plot.barh(ax=ax[1], title='Sintagmas nominais mais frequentes')
plt.tight_layout()

**Termos emergentes:** comparação da frequência relativa entre a primeira e a segunda metade do período.

In [ ]:
meio = (ANO_INICIAL + ANO_FINAL) // 2
por_periodo = (df.assign(periodo=df['ano'].le(meio).map({True: 'inicio', False: 'fim'}))
                 .explode('lemas')
                 .groupby(['lemas', 'periodo']).size().unstack(fill_value=0))
por_periodo = por_periodo.div(por_periodo.sum())     # frequência relativa em cada metade
por_periodo['variacao'] = por_periodo['fim'] - por_periodo['inicio']

emergentes = por_periodo.sort_values('variacao', ascending=False).head(10)
em_queda = por_periodo.sort_values('variacao').head(10)
pd.DataFrame({'emergente': emergentes.index, 'aumento': emergentes['variacao'].round(4).values,
              'em queda': em_queda.index, 'redução': em_queda['variacao'].round(4).values})

**Entidades nomeadas** mais citadas (locais, organizações e outras).

In [ ]:
entidades = (df.explode('entidades').dropna(subset=['entidades'])
                .assign(texto=lambda d: d['entidades'].str[0], tipo=lambda d: d['entidades'].str[1])
                .groupby(['tipo', 'texto']).size().rename('n')
                .sort_values(ascending=False).reset_index())
entidades.head(15)

## 5. Modelagem de tópicos (TF-IDF + NMF)

Com a base real, teste alguns valores de `N_TOPICOS` e escolha o que gera tópicos mais interpretáveis. O BERTopic é uma alternativa baseada em *embeddings* (veja a seção 8).

In [ ]:
textos_lematizados = df['lemas'].str.join(' ')
vetorizador = TfidfVectorizer(max_df=0.9, min_df=1 if len(df) < 50 else 3, ngram_range=(1, 2))
X = vetorizador.fit_transform(textos_lematizados)

nmf = NMF(n_components=N_TOPICOS, random_state=42, init='nndsvda', max_iter=500)
W = nmf.fit_transform(X)
vocab = vetorizador.get_feature_names_out()

topicos = {f'T{i}': ', '.join(vocab[j] for j in comp.argsort()[::-1][:8])
           for i, comp in enumerate(nmf.components_)}
df['topico'] = [f'T{i}' for i in W.argmax(axis=1)]
pd.Series(topicos, name='termos principais').to_frame()

In [ ]:
participacao = pd.crosstab(df['ano'], df['topico'], normalize='index')
participacao.plot.area(figsize=(10, 4), title='Participação dos tópicos por ano')
plt.ylabel('proporção dos artigos')
plt.tight_layout()

## 6. Cautela científica (*hedging*)

Contagem de expressões de cautela (*hedges*) e de ênfase (*boosters*) por 100 palavras. O léxico abaixo é um ponto de partida: revise-o com exemplos reais da sua área.

In [ ]:
HEDGES = ['sugerir', 'indicar', 'parecer', 'poder', 'possível', 'possivelmente',
          'provavelmente', 'aparentemente', 'hipótese', 'tendência', 'eventualmente']
BOOSTERS = ['demonstrar', 'comprovar', 'evidenciar', 'confirmar', 'claramente',
            'certamente', 'inequivocamente', 'significativo']

matcher = Matcher(nlp.vocab)
matcher.add('HEDGE', [[{'LEMMA': {'IN': HEDGES}}]])
matcher.add('BOOSTER', [[{'LEMMA': {'IN': BOOSTERS}}]])

def contar(doc):
    rotulos = [nlp.vocab.strings[m_id] for m_id, _, _ in matcher(doc)]
    return rotulos.count('HEDGE'), rotulos.count('BOOSTER')

df[['n_hedges', 'n_boosters']] = [contar(d) for d in docs]
df['hedging_100'] = 100 * df['n_hedges'] / df['n_tokens']
df['enfase_100'] = 100 * df['n_boosters'] / df['n_tokens']

df.groupby('ano')[['hedging_100', 'enfase_100']].mean().plot(
    marker='o', figsize=(10, 4), title='Cautela e ênfase por ano (média por 100 palavras)')
plt.tight_layout()

In [ ]:
df.groupby('topico')[['hedging_100', 'enfase_100']].mean().round(2)

## 7. Exportação dos resultados

In [ ]:
df.drop(columns=['resumo']).to_csv(PASTA_RESULTADOS / 'artigos_anotados.csv', index=False)
pd.Series(topicos).to_csv(PASTA_RESULTADOS / 'topicos.csv', header=['termos'])
participacao.to_csv(PASTA_RESULTADOS / 'participacao_topicos_por_ano.csv')
df[['doi']].to_csv('../DATA/dois.csv', index=False)  # lista publicável para reconstruir a base
print('Resultados salvos em', PASTA_RESULTADOS.resolve())

## 8. Alternativa: BERTopic

Para bases maiores (algumas centenas de artigos ou mais), o BERTopic costuma gerar tópicos mais coerentes. Ele não está no `requirements.txt` porque é pesado; instale com `pip install bertopic` e descomente a célula.

In [ ]:
# from bertopic import BERTopic
# from sentence_transformers import SentenceTransformer
#
# modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
# topic_model = BERTopic(embedding_model=modelo_emb, language='multilingual')
# rotulos, _ = topic_model.fit_transform(df['resumo'].tolist())
# topic_model.get_topic_info()